# Comparação de modelos de embeddings

Recall@k e MRR por modelo × categoria de reescrita, com o conjunto de teste do 05 (`results/teste_reescrita.csv`).

**`INDICE`:** `"2022"` = preliminar (1.852 alegações); `"ambos"` = **avaliação definitiva** (2022 + Central de Fatos, ~12 mil alegações), a rodar depois da rotulagem entre ciclos. Os acertos são definidos pelo texto da alegação, então o mesmo conjunto de teste vale nos dois índices.

**Regra de escolha:** pré-registrada no `design.md` antes da rodada definitiva.

**Consulta:** avaliada `bruta` e com a normalização `condicional` (apelidos sempre; Enelvo só quando a consulta tem cara de internetês), a melhor do 05.

Saídas: `results/comparacao_modelos_<INDICE>.csv`; embeddings do índice em cache em `data/emb/` (DVC).

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd

from avaliacao import linhas_com_texto, metricas, posicoes
from corpora import carregar_indice
from modelos import CACHE, MODELOS, carregar, embeddings
from normalizacao import criar_enelvo, normalizar_se_preciso, vocabulario_protegido
from reescrita import carregar_apelidos, carregar_internetes

INDICE = "2022"          # "2022" (preliminar) ou "ambos" (definitiva)
RESULTS = Path("results")
CACHE.mkdir(parents=True, exist_ok=True)

indice = carregar_indice(INDICE)
teste = pd.read_csv(RESULTS / "teste_reescrita.csv").fillna({"excluir_texto": ""})
print(f"índice {INDICE}: {len(indice)} alegações ({indice['origem'].value_counts().to_dict()}) | {len(teste)} consultas")

## Consultas: bruta e normalizada

O vocabulário protegido do Enelvo vem do índice, então muda com o `INDICE`.

In [ ]:
apelidos, tabela_internetes = carregar_apelidos(), carregar_internetes()
enelvo = criar_enelvo(vocabulario_protegido(indice["claim"], tabela_internetes), CACHE / f"protegidos_{INDICE}.txt")
teste["condicional"] = [normalizar_se_preciso(t, enelvo, apelidos, tabela_internetes) for t in teste["consulta"]]

relevantes = linhas_com_texto(indice["claim"], teste["alvo"])
excluir = linhas_com_texto(indice["claim"], teste["excluir_texto"])
assert all(len(r) for r in relevantes), "alvo fora do índice"

## Rodar os modelos

Para cada modelo: embeddings do índice (do cache, se já existirem para esta assinatura), embeddings das consultas, posição do alvo e métricas por categoria. O tempo de codificação entra no resultado: custo de deploy.

In [ ]:
linhas = []
for familia, nome, revisao, prefixo in MODELOS:
    model = carregar(nome, revisao)
    E, seg_por_mil = embeddings(indice["claim"], nome, revisao, prefixo)   # cache em data/emb/ pelo hash dos textos
    for variante in ["consulta", "condicional"]:
        Q = model.encode([prefixo + t for t in teste[variante]], normalize_embeddings=True, batch_size=32)
        pos = posicoes(Q, E, relevantes, excluir)
        for categoria, g in teste.assign(pos=pos).groupby("categoria"):
            linhas.append({"familia": familia, "modelo": nome, "variante": "bruta" if variante == "consulta" else variante,
                           "categoria": categoria, **metricas(g["pos"].to_numpy()), "seg_por_mil_textos": seg_por_mil})
    print(f"✓ {nome}")

resultado = pd.DataFrame(linhas)
resultado.insert(0, "indice", INDICE)
resultado.to_csv(RESULTS / f"comparacao_modelos_{INDICE}.csv", index=False)

## Resumo

Recall@5 por categoria, com a média entre categorias (macro: cada categoria pesa igual, independente do n).

In [ ]:
for variante in ["bruta", "condicional"]:
    r = resultado[resultado["variante"] == variante]
    tabela = r.pivot(index="modelo", columns="categoria", values="recall@5")
    tabela["média"] = tabela.mean(axis=1)
    print(f"\n=== Recall@5, consulta {variante} ===")
    display(tabela.sort_values("média", ascending=False).round(2))